In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [3]:
from src.pipeline import load_incremental_orders, get_database_connection, clean_sales_data

In [4]:
import duckdb

In [5]:
con = duckdb.connect(":memory:")

In [6]:
con.execute("""BEGIN""")

In [7]:
con.execute("""COMMIT""")

In [8]:
con.execute("""ROLLBACK""")

TransactionException: TransactionContext Error: cannot rollback - no transaction is active

In [9]:
import pandas as pd

In [10]:
test_orders_df = pd.DataFrame({"order_id":[2001,2002],
                               "product":["Laptop","Monitor"],
                               "quantity":[1,2],
                               "price":[1200,300],
                               "order_date":["2026-02-01","2026-02-02"]})

In [11]:
clean_test_orders_df = clean_sales_data(test_orders_df)

In [12]:
con.execute("""BEGIN""")

In [13]:
load_incremental_orders(clean_test_orders_df,con)

In [14]:
con.execute("""
    SELECT *
    FROM table_that_does_not_exist
""")

CatalogException: Catalog Error: Table with name table_that_does_not_exist does not exist!
Did you mean "information_schema.tables"?

LINE 3:     FROM table_that_does_not_exist
                 ^

In [15]:
con.execute("""ROLLBACK""")

In [16]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM clean_orders
    """).df()

CatalogException: Catalog Error: Table with name clean_orders does not exist!
Did you mean "pg_indexes"?

LINE 3:     FROM clean_orders
                 ^

In [17]:
empty_orders_df = clean_test_orders_df.head(0)

In [18]:
load_incremental_orders(empty_orders_df,con)

In [19]:
con.execute("BEGIN")

In [20]:
load_incremental_orders(clean_test_orders_df,con)

In [21]:
con.execute("ROLLBACK")

In [22]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM clean_orders
    """).df()

,row_count
0,0


In [23]:
def run_transaction_safe_load(clean_df,con):
    try:
        con.execute("BEGIN")
        load_incremental_orders(clean_df,con)
        con.execute("COMMIT")
    except Exception as e:
        con.execute("ROLLBACK")
        raise

In [24]:
run_transaction_safe_load(clean_test_orders_df,con)

In [25]:
con.execute("""
    SELECT
        COUNT(*) AS row_count
    FROM clean_orders
""").df()

,row_count
0,2


In [26]:
run_transaction_safe_load(clean_test_orders_df,con)

In [27]:
con.execute("""
    SELECT
        COUNT(*) AS row_count
    FROM clean_orders
""").df()

,row_count
0,2


In [28]:
failure_orders_df = pd.DataFrame({"order_id":[2003],
              "product":["Mouse"],
              "quantity":[2],
              "price":[50],
              "order_date":["2026-02-03"]})

In [29]:
clean_failure_orders_df = clean_sales_data(failure_orders_df)

In [30]:
def run_transaction_failure_test(clean_df,con):
    try:
        con.execute("BEGIN")
        load_incremental_orders(clean_df,con)
        con.execute("""
            SELECT
                *
            FROM table_that_does_not_exist
        """)
    except Exception as e:
        con.execute("ROLLBACK")
        raise

In [31]:
run_transaction_failure_test(clean_failure_orders_df,con)

CatalogException: Catalog Error: Table with name table_that_does_not_exist does not exist!
Did you mean "information_schema.tables"?

LINE 4:             FROM table_that_does_not_exist
                         ^

In [32]:
con.execute("""
    SELECT
        COUNT(*) AS row_count
    FROM clean_orders
""").df()

,row_count
0,2


In [33]:
incoming_count = len(clean_test_orders_df)

In [34]:
failure_batch_count = len(clean_failure_orders_df)

In [35]:
before_count = con.execute("""
    SELECT
        COUNT(*)
    FROM clean_orders
""").fetchone()[0]

In [36]:
before_count

2

In [37]:
run_transaction_safe_load(clean_failure_orders_df, con)

In [38]:
after_count = con.execute("""
    SELECT
        COUNT(*)
    FROM clean_orders
""").fetchone()[0]

In [39]:
after_count

3

In [40]:
inserted_count = after_count - before_count

In [42]:
display(inserted_count)

1

In [43]:
run_transaction_safe_load(clean_failure_orders_df, con)

In [44]:
repeat_after_count = con.execute("""
    SELECT
        COUNT(*)
    FROM clean_orders
""").fetchone()[0]

In [45]:
repeat_inserted_count = repeat_after_count - after_count

In [46]:
import time

In [47]:
start_time = time.perf_counter()

In [48]:
run_transaction_safe_load(clean_failure_orders_df, con)

In [49]:
end_time = time.perf_counter()

In [51]:
duration_seconds = end_time - start_time

In [52]:
duration_seconds

97.08180650000213

In [53]:
start_time = time.perf_counter()
run_transaction_safe_load(clean_failure_orders_df, con)
end_time = time.perf_counter()

In [54]:
duration_seconds = end_time - start_time

In [55]:
duration_seconds

0.003926000001229113

In [58]:
pipeline_metrics = {"incoming_rows":failure_batch_count,
                    "inserted_rows":repeat_inserted_count,
                    "duration_seconds":duration_seconds,
                    "status":"success"}

In [59]:
display(pipeline_metrics)

{'incoming_rows': 1,
 'inserted_rows': 0,
 'duration_seconds': 0.003926000001229113,
 'status': 'success'}

In [60]:
pipeline_metrics["status"]

'success'

In [61]:
pipeline_metrics["inserted_rows"]

0

In [62]:
import logging

In [63]:
logging.info("Pipeline completed successfully")

2026-09-22 21:16:49,393 - INFO - Pipeline completed successfully


In [64]:
logging.error("Pipeline load failed")

2026-09-22 21:17:12,560 - ERROR - Pipeline load failed


In [ ]:
logging.info(f"Inserted rows: {pipeline_metrics['inserted_rows']}")

2026-09-22 21:26:01,096 - INFO - Inserted rows: 0


In [66]:
logging.info(f"Pipeline duration: {pipeline_metrics['duration_seconds']}")

2026-09-22 21:26:56,882 - INFO - Pipeline duration: 0.003926000001229113


In [67]:
logging.info(pipeline_metrics)

2026-09-22 21:27:22,538 - INFO - {'incoming_rows': 1, 'inserted_rows': 0, 'duration_seconds': 0.003926000001229113, 'status': 'success'}


In [68]:
logging.info(f"Incoming rows: {pipeline_metrics['incoming_rows']}")

2026-09-22 21:29:53,706 - INFO - Incoming rows: 1


In [69]:
logging.basicConfig(level=logging.INFO)

In [70]:
logging.info("Day 15 monitoring is ready")

2026-09-22 21:31:00,905 - INFO - Day 15 monitoring is ready


In [74]:
from src.pipeline import run_transaction_safe_load

In [73]:
import importlib
import src.pipeline

importlib.reload(src.pipeline)

<module 'src.pipeline' from 'c:\\Users\\gilbe\\OneDrive\\Desktop\\30 Day DS DE\\src\\pipeline.py'>

In [75]:
run_transaction_safe_load(clean_failure_orders_df, con)

In [ ]:
verified_count = con.execute("""
                    SELECT
                        COUNT(*) AS verified_count
                    FROM clean_orders
                    """).fetchone()[0]